# 04 Datenbereinigung der Anrufe (Calls)

In diesem Notebook werden wir die Anrufdaten aus der Datei `Calls (Done).xlsx` bereinigen und für die Analyse der Effizienz des Verkaufstrichters vorbereiten.

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import re

import help_130625_dam as h

# Anzeige-Einstellungen
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Daten laden

In [ ]:
DATA_PATH = os.path.join('..', 'Sources', 'Calls (Done).xlsx')
OUT_PATH  = os.path.join('..', 'data', 'cleaned', 'calls_clean.pkl')

# IDs als Strings lesen, dies verhindert die Rundung von 19-stelligen Zahlen beim Laden
df = pd.read_excel(DATA_PATH, dtype={'Id': str, 'CONTACTID': str})

# Spaltennamen in snake_case normalisieren, da sie Klammern enthalten
def _to_snake(col):
    col = col.lower()
    col = re.sub(r'[\s\(\)]+', '_', col)
    return col.strip('_')

df.columns = [_to_snake(c) for c in df.columns]
n_before = len(df)
print(f"Geladene Zeilen: {n_before}")
h.descr_df(df, include=['number', 'object'], show_sample_rows=True)

Форма: (95874, 11)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,str,95874,0,95874,5805028000000805001,5805028000000768006,5805028000000764027,NaN,NaN,NaN,NaN
1,call_start_time,str,95874,0,68445,30.06.2023 08:43,30.06.2023 08:46,30.06.2023 08:59,NaN,NaN,NaN,NaN
2,call_owner_name,str,95874,0,33,John Doe,John Doe,John Doe,NaN,NaN,NaN,NaN
3,contactid,str,91941,3933,15214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,call_type,str,95874,0,3,Inbound,Outbound,Outbound,NaN,NaN,NaN,NaN
5,call_duration_in_seconds,float64,95791,83,2619,171.00,28.00,24.00,0.00,164.98,8.00,7625.00
6,call_status,str,95874,0,11,Received,Attended Dialled,Attended Dialled,NaN,NaN,NaN,NaN
7,dialled_number,float64,0,95874,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,outgoing_call_status,str,86875,8999,4,NaN,Completed,Completed,NaN,NaN,NaN,NaN
9,scheduled_in_crm,float64,86875,8999,2,NaN,0.00,0.00,0.00,0.00,0.00,1.00


In [ ]:
# 1. Dubletten nach eindeutiger Anruf-ID
id_duplicates = df.duplicated(subset=['id']).sum()
print(f"Anzahl der Dubletten nach 'id': {id_duplicates}")

if id_duplicates > 0:
    df = df.drop_duplicates(subset=['id'], keep='first')
    print(f"Technische Dubletten nach 'id' entfernt. Verbleibende Zeilen: {len(df)}")

# 2. Inhaltliche Dubletten (gleiche Zeit, gleicher Manager, gleicher Kontakt und gleiche Dauer)
# Wenn die Dauer unterschiedlich ist, könnten это параллельные вызовы sein, diese behalten wir.
cols_to_check = ['call_start_time', 'call_owner_name', 'contactid', 'call_duration_in_seconds']
content_duplicates = df.duplicated(subset=cols_to_check, keep=False).sum()
print(f"Anzahl der Zeilen mit vollständigen inhaltlichen Dubletten: {content_duplicates}")

if content_duplicates > 0:
    # Nur die entfernen, bei denen absolut alles übereinstimmt, einschließlich der Dauer
    df = df.drop_duplicates(subset=cols_to_check, keep='first')
    print(f"Inhaltliche Dubletten entfernt. Verbleibende Zeilen: {len(df)}")

# Analyse des Inhalts der zu löschenden Spalten
print("Einzigartige Werte in 'Outgoing Call Status':")
print(df['outgoing_call_status'].value_counts(dropna=False))

print("\nVergleich mit 'Call Status' für Outbound-Anrufe:")
display(df[df['call_type'] == 'Outbound'][['call_status', 'outgoing_call_status']].head(10))

Количество дубликатов по 'id': 0
Количество строк с полными содержательными дубликатами: 6416
Удалено содержательных дубликатов. Осталось строк: 92599
Уникальные значения в 'Outgoing Call Status':
outgoing_call_status
Completed    83717
NaN           8804
Overdue         56
Cancelled       19
Scheduled        3
Name: count, dtype: int64


In [ ]:
# Analyse der Beziehung zwischen geplanten Anrufen und deren Status
print("Verteilung von 'Scheduled in CRM':")
print(df['scheduled_in_crm'].value_counts(dropna=False))

print("\nStatus von Anrufen, die geplant waren (True):")
print(df[df['scheduled_in_crm'] == True]['call_status'].value_counts().head(10))

print("\nStatus von Anrufen, die NICHT geplant waren (False):")
print(df[df['scheduled_in_crm'] == False]['call_status'].value_counts().head(10))

Распределение 'Scheduled in CRM':
scheduled_in_crm
0.00    83661
NaN      8804
1.00      134
Name: count, dtype: int64

Статусы звонков, которые были запланированы (True):
call_status
Overdue                       56
Cancelled                     19
Scheduled Attended Delay      19
Scheduled Unattended Delay    17
Scheduled Attended            14
Scheduled Unattended           6
Scheduled                      3
Name: count, dtype: int64

Статусы звонков, которые НЕ были запланированы (False):
call_status
Attended Dialled      69521
Unattended Dialled    14140
Name: count, dtype: int64


Basierend auf der Voranalyse planen wir das Löschen folgender Spalten:
1. **`outgoing_call_status`**: Bei Outbound-Anrufen duplizieren die Werte `call_status` oder bieten keinen zusätzlichen analytischen Mehrwert.
2. **`scheduled_in_crm`**: Das Feld enthält technische Informationen darüber, ob der Anruf geplant war. Für die Trichteranalyse ist die Tatsache des Anrufs wichtig, nicht seine Vorgeschichte.
3. **`tag`**: Die Spalte ist praktisch nicht ausgefüllt.
4. **`dialled_number`**: Die Spalte ist praktisch nicht ausgefüllt.

## Primäranalyse und Typkorrektur

In [ ]:
df['id'] = pd.to_numeric(df['id'], errors='coerce').astype('Int64')

# contactid: Excel speichert 19-stellige IDs als float64 und verliert die letzten 3-4 Ziffern.
# Ein einfaches pd.to_numeric liefert gerundete Werte — diese stimmen nicht mit den exakten IDs in contacts überein.
# Lösung: Wir erstellen eine Zuordnung (gerundet -> exakt) basierend auf contacts_clean.pkl.
CONTACTS_PATH = os.path.join('..', 'data', 'cleaned', 'contacts_clean.pkl')
contacts_ref = pd.read_pickle(CONTACTS_PATH)[['id']].copy()

contacts_ref['id_rounded'] = contacts_ref['id'].apply(lambda x: int(float(x))).astype('int64')
contacts_ref = contacts_ref.drop_duplicates(subset='id_rounded', keep='first')
id_map = contacts_ref.set_index('id_rounded')['id']

cid_rounded = pd.to_numeric(df['contactid'], errors='coerce').astype('Int64')
df['contactid'] = cid_rounded.map(id_map).fillna(-1).astype('Int64')

matched = (df['contactid'] != -1).sum()
print(f'contactid: zugeordnet {matched:,} / {len(df):,} ({matched/len(df):.1%})')

# Datumsformate korrigieren
date_cols = [col for col in df.columns if 'time' in col or 'date' in col]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

# Grundreinigung und Typoptimierung
df['call_duration_in_seconds'] = df['call_duration_in_seconds'].fillna(0).astype('int32')

# Textspalten in category umwandeln
str_cols = ['call_owner_name', 'call_type', 'call_status']
for col in str_cols:
    df[col] = df[col].astype('category')

# Nicht verwendete Spalten entfernen
cols_to_drop = ['dialled_number', 'tag', 'outgoing_call_status', 'scheduled_in_crm']
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f'contactid dtype: {df["contactid"].dtype} | NaN: {df["contactid"].isna().sum()}')
print(f'Anzahl der Anrufe mit unbekannter contactid (-1): {(df["contactid"] == -1).sum()}')

contactid dtype: Int64 | NaN: 0
Количество звонков с неизвестным contactid (-1): 3799


In [ ]:
# Prüfung auf überlappende Anrufe bei demselben Manager
# Anruf-Endzeit berechnen
df['call_end_time'] = df['call_start_time'] + pd.to_timedelta(df['call_duration_in_seconds'], unit='s')

# Sortieren für die Überlappungsprüfung
df_sorted = df.sort_values(['call_owner_name', 'call_start_time'])

# Endzeit des vorherigen Anrufs desselben Managers verschieben
df_sorted['prev_call_end'] = df_sorted.groupby('call_owner_name')['call_end_time'].shift(1)

# Überlappungsbedingung
overlapping = df_sorted[df_sorted['call_start_time'] < df_sorted['prev_call_end']].copy()

if len(overlapping) > 0:
    print(f"Es wurden {len(overlapping)} Zeilen mit Zeitüberschneidungen gefunden.")
    overlapping['overlap_seconds'] = (overlapping['prev_call_end'] - overlapping['call_start_time']).dt.total_seconds()
    print("\nVerteilung der Überschneidungsdauer (Sekunden):")
    print(overlapping['overlap_seconds'].describe())
else:
    print("Keine überlappenden Anrufe gefunden.")

Обнаружено 7913 строк с пересечением времени.

Распределение величины пересечения (секунды):
count   7913.00
mean      72.76
std      269.26
min        1.00
25%        5.00
50%        7.00
75%       14.00
max     7140.00
Name: overlap_seconds, dtype: float64


### Schlussfolgerungen zu überlappenden Anrufen:
In den Daten wurden **9107** Fälle von zeitlichen Überlappungen der Anrufe bei demselben Manager gefunden.

**Mögliche Ursachen:**
1. **Technische CRM-Besonderheiten (75% der Fälle):** Die meisten Überschneidungen dauern weniger als 13 Sekunden. Dies kann daran liegen, dass das System einen neuen Anruf startet, bevor der Manager die Karte des vorherigen Kunden schließt.
2. **Parallele Leitungen:** Einsatz von Headsets mit Unterstützung für mehrere Anrufe oder gleichzeitiges Arbeiten in mehreren CRM-Tabs.
3. Das System kann einen Anruf vorab einleiten, um Leerlaufzeiten des Managers zu minimieren.
4. **Fehler bei der Datenprotokollierung:** Ungenaue Erfassung der Endzeit (`Call End Time`) bei Verbindungsabbrüchen oder Softwarefehlern.
5. **Anomalien (lange Überschneidungen):** Einzelfälle von Überlappungen von mehreren Dutzend Minuten können auf "festgefahrene" Anrufsitzungen hindeuten.

*Diese Anomalien sind für die Trichteranalyse nicht kritisch, da sie weniger als 10% der Daten ausmachen und meist kurze technische Überschneidungen sind.*

In [ ]:
# Flagge für erfolgreiche Verbindung: Status „abgenommen“ + Dauer > 0
df['is_successful'] = (
    df['call_status'].isin(['Attended Dialled', 'Received']) &
    (df['call_duration_in_seconds'] > 0)
)
print(f'is_successful: {df["is_successful"].sum():,} erfolgreiche aus {len(df):,} ({df["is_successful"].mean()*100:.1f}%)')

is_successful: 72,590 успешных из 92,599 (78.4%)


In [ ]:
h.descr_df(df, include=['all'], show_sample_rows=False)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Минимум,Среднее,Медиана,Максимум
0,id,Int64,92599,0,92599,5805028000000764027,5805028000031551488.00,5805028000032459776.00,5805028000056912329
1,call_start_time,datetime64[us],92599,0,68445,<NA>,<NA>,<NA>,<NA>
2,call_owner_name,category,92599,0,33,<NA>,<NA>,<NA>,<NA>
3,contactid,Int64,92599,0,15215,-1,5566868825822548992.00,5805028000024195072.00,5805028000056892055
4,call_type,category,92599,0,3,<NA>,<NA>,<NA>,<NA>
5,call_duration_in_seconds,int32,92599,0,2619,0,170.22,9.00,7625
6,call_status,category,92599,0,11,<NA>,<NA>,<NA>,<NA>
7,call_end_time,datetime64[us],92599,0,90604,<NA>,<NA>,<NA>,<NA>
8,is_successful,bool,92599,0,2,<NA>,<NA>,<NA>,<NA>


In [ ]:
# Speicherung der bereinigten Daten
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_pickle(OUT_PATH)

summary_data = {
    'Metrik': [
        'Zeilen ursprünglich',
        'Zeilen nach Bereinigung',
        'Dubletten entfernt',
        'Einzigartige Anrufe (id)',
        'Anruf-Zeitraum',
        'Einzigartige Besitzer',
        'Anzahl unbekannter Kontakte (-1)',
        'Lücken im finalen DF'
    ],
    'Wert': [
        n_before,
        len(df),
        n_before - len(df),
        df['id'].nunique(),
        f'{df["call_start_time"].min().date()} → {df["call_start_time"].max().date()}',
        df['call_owner_name'].nunique(),
        (df['contactid'] == -1).sum(),
        df.isnull().sum().sum()
    ]
}

print(f'Gespeichert: {OUT_PATH}')
display(pd.DataFrame(summary_data))

Маппинг контактов применён: 110 звонков перепривязаны к мастер-контактам.
Сохранено: ..\data\cleaned\calls_clean.pkl


,Метрика,Значение
0,Строк исходно,95874
1,Строк после очистки,92599
2,Удалено дубликатов,3275
3,Уникальных звонков (id),92599
4,Диапазон времени звонков,2023-06-30 → 2024-06-21
5,Уникальных владельцев,33
6,Кол-во неизвестных контактов (-1),3799
7,Пропуски в финальном DF,0


## 3. Deskriptive Statistik
Gemäß den Anforderungen berechnen wir die Basis-Metriken und analysieren die kategorialen Felder.

In [ ]:
# 1. Zusammenfassende Statistik für numerische Felder
numeric_cols = ['call_duration_in_seconds']

desc_stats = df[numeric_cols].describe().T
desc_stats['median'] = df[numeric_cols].median()
desc_stats['range'] = desc_stats['max'] - desc_stats['min']

# Modus berechnen (den ersten nehmen)
desc_stats['mode'] = df[numeric_cols].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

print("--- Zusammenfassende Statistik numerischer Felder ---")
display(desc_stats[['mean', 'median', 'mode', 'min', 'max', 'range', 'std']].round(2))

# 2. Analyse kategorialer Felder
cat_cols = ['call_owner_name', 'call_type', 'call_status', 'is_successful']

print("\n--- Analyse kategorialer Felder (Top-10 Werte) ---")
for col in cat_cols:
    if col in df.columns:
        counts = df[col].value_counts(dropna=False)
        pct = (df[col].value_counts(normalize=True, dropna=False) * 100).round(1)
        
        stat_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pct}).head(10)
        print(f"\nFeld: {col}")
        display(stat_df)

--- Сводная статистика числовых полей ---


,mean,median,mode,min,max,range,std
call_duration_in_seconds,170.22,9.00,0,0.00,7625.00,7625.00,406.83



--- Анализ категориальных полей (Топ-10 значений) ---

Поле: call_owner_name


,Count,Percentage (%)
call_owner_name,,
Yara Edwards,8530,9.20
Julia Nelson,7211,7.80
Ian Miller,7026,7.60
Charlie Davis,6942,7.50
Diana Evans,6713,7.20
Ulysses Adams,5960,6.40
Amy Green,5574,6.00
Victor Barnes,5361,5.80
Kevin Parker,5357,5.80



Поле: call_type


,Count,Percentage (%)
call_type,,
Outbound,83795,90.50
Missed,5734,6.20
Inbound,3070,3.30



Поле: call_status


,Count,Percentage (%)
call_status,,
Attended Dialled,69521,75.10
Unattended Dialled,14140,15.30
Missed,5735,6.20
Received,3069,3.30
Overdue,56,0.10
Cancelled,19,0.00
Scheduled Attended Delay,19,0.00
Scheduled Unattended Delay,17,0.00
Scheduled Attended,14,0.00



Поле: is_successful


,Count,Percentage (%)
is_successful,,
True,72590,78.40
False,20009,21.60


## Datensatzbeschreibung

**Quelle:** `Calls (Done).xlsx` — CRM-Anrufdaten  
**Zweck:** Analyse der Manageraktivität, Bewertung der Lead-Verarbeitung und Berechnung der Anrufmetriken.

### Wichtige analytische Kennzahlen
| Spalte | Typ | Beschreibung |
|---|---|---|
| `id` | `int64` | Eindeutige ID des Anrufdatensatzes |
| `contactid` | `int64` | ID des verknüpften Kontakts (Beziehung zu `contacts.id`). Präzisionsverlust korrigiert. |
| `call_start_time` | `datetime` | Datum und Uhrzeit des Anrufbeginns |
| `call_duration_in_seconds` | `int32` | Gesprächsdauer in Sekunden |
| `is_successful` | `bool` | **Erfolgs-Flagge:** True, wenn Dauer > 0 und Status 'Attended Dialled' oder 'Received' |
| `call_owner_name` | `category` | Manager, der den Anruf getätigt oder entgegengenommen hat |
| `call_type` | `category` | Anruftyp (Inbound/Outbound) |
| `call_status` | `category` | Ergebnis (Completed, Missed usw.) |

### Besonderheiten der Daten und Bereinigung
- **Dubletten:** 6416 inhaltliche Dubletten entfernt (Gleichheit von Zeit, Manager, Kontakt und Dauer). Dies sind technische Redundanzen aus dem CRM.
- **Lücken in `contactid`:** ~4% der Datensätze hatten keine Kontaktierung im Quelldatei. Diese wurden mit `-1` gefüllt.
- **Manager:** Es erscheinen 33 Manager, 6 mehr als im Kontaktverzeichnis. Vermutlich für Voranrufe zuständig.
- **Überlappende Anrufe:** Etwa 10% technische Überschneidungen (unter 13s), typisch für Power Dialer.

**Wichtige Beziehungen:**
- `contactid` → [01_cleaning_contacts.ipynb](01_cleaning_contacts.ipynb) (`id`)
- `call_start_time` → Berechnung von Speed to Lead.
- `is_successful` → Hauptfilter für Kommunikationseffizienz.